# Use API for analysis rather than downloading as storage
- https://www.tycho.pitt.edu/dataset/api/

In [1]:
import pandas as pd

nis_vacc_coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
nis_vacc_coverage_df['year'].min(), nis_vacc_coverage_df['year'].max()

(np.int64(1995), np.int64(2024))

In [2]:
import os
from dotenv import load_dotenv

# Load variables from .env file into the environment
load_dotenv()

# Access the variables
API_KEY = os.getenv("TYCHO_API_KEY")

### Columns needed from this query:
- ConditionName
- ConditionSNOMED
- CountryCode
- Admin1ISO
- Admin1Name
- PeriodStartDate
- PeriodEndDate
- CountValue
- PartOfCumulativeCountSeries
- Fatalities

In [3]:
core_columns = [
    "ConditionName",
    "Admin1ISO",
    "PeriodEndDate",
    "CountValue",
    "PartOfCumulativeCountSeries",
]

In [4]:
import pandas as pd

diseases = ["Measles", "Mumps", "Pertussis"]

disease_dfs = []

for disease in diseases:

    print(f"\nDownloading {disease}...")

    dfs = []
    offset = 0
    limit = 5000

    while True:

        url = (
            f"https://www.tycho.pitt.edu/api/query?"
            f"apikey={API_KEY}"
            f"&ConditionName={disease}"
            f"&CountryISO=US"
            f"&PeriodStartDate%3E=1995-01-01"
            f"&PeriodEndDate%3C=2024-12-31"
            f"&limit={limit}"
            f"&offset={offset}"
        )

        temp_df = pd.read_csv(url)

        # Stop when there is no more data
        if not set(core_columns).issubset(temp_df.columns):
            break

        temp_df = temp_df[core_columns]

        dfs.append(temp_df)

        print(
            f"{disease}: downloaded {len(temp_df):,} rows "
            f"| offset: {offset:,}"
        )

        # Stop if this was the last page
        if len(temp_df) < limit:
            break

        offset += limit

    # Combine pages for this disease
    df = pd.concat(dfs, ignore_index=True)

    # Clean
    df["year"] = (
        df["PeriodEndDate"]
        .str.split("-")
        .str[0]
        .astype(int)
    )

    df["state"] = (
        df["Admin1ISO"]
        .str.split("-")
        .str[1]
        .astype(str)
    )

    df.drop(
        columns=[
            "Admin1ISO",
            "PeriodEndDate",
        ],
        inplace=True
    )

    print(f"{disease} final: {df.shape}")

    disease_dfs.append(df)


# Combine all three diseases
df = pd.concat(
    disease_dfs,
    ignore_index=True
)

print("\nCombined:", df.shape)

df.head()


Measles: downloaded 5,000 rows | offset: 0
Measles: downloaded 5,000 rows | offset: 5,000
Measles: downloaded 3,495 rows | offset: 10,000
Measles final: (13495, 5)

Mumps: downloaded 5,000 rows | offset: 0
Mumps: downloaded 5,000 rows | offset: 5,000
Mumps: downloaded 5,000 rows | offset: 10,000
Mumps: downloaded 5,000 rows | offset: 15,000
Mumps: downloaded 5,000 rows | offset: 20,000
Mumps: downloaded 563 rows | offset: 25,000
Mumps final: (25563, 5)

Pertussis: downloaded 5,000 rows | offset: 0
Pertussis: downloaded 5,000 rows | offset: 5,000
Pertussis: downloaded 5,000 rows | offset: 10,000
Pertussis: downloaded 5,000 rows | offset: 15,000
Pertussis: downloaded 5,000 rows | offset: 20,000
Pertussis: downloaded 5,000 rows | offset: 25,000
Pertussis: downloaded 5,000 rows | offset: 30,000
Pertussis: downloaded 5,000 rows | offset: 35,000
Pertussis: downloaded 5,000 rows | offset: 40,000
Pertussis: downloaded 5,000 rows | offset: 45,000
Pertussis: downloaded 5,000 rows | offset: 50,0

,ConditionName,CountValue,PartOfCumulativeCountSeries,year,state
0,Measles,1,0,1995,OH
1,Measles,1,0,1995,OH
2,Measles,2,0,1996,OH
3,Measles,3,0,1996,OH
4,Measles,1,0,1996,OH


### Filter out territries

In [8]:
territories = ["PR", "VI", "GU", "AS", "MP"]

df = df[~df["state"].isin(territories)]
len(df)

108713

### Aggregate

In [11]:
group_columns = [
    "year",
    "state",
    "ConditionName"
]

# Sum individual-period case counts
non_cumulative_df = (
    df[df["PartOfCumulativeCountSeries"] == 0]
    .groupby(group_columns, as_index=False)
    .agg(
        cases=("CountValue", "sum"),
        records=("CountValue", "size")
    )
)

non_cumulative_df["reporting_type"] = "non_cumulative"


# Use the largest cumulative value as the annual endpoint
cumulative_df = (
    df[df["PartOfCumulativeCountSeries"] == 1]
    .groupby(group_columns, as_index=False)
    .agg(
        cases=("CountValue", "max"),
        records=("CountValue", "size")
    )
)

cumulative_df["reporting_type"] = "cumulative"

### Combine

In [12]:
state_year_cases_long = (
    pd.concat(
        [cumulative_df, non_cumulative_df],
        ignore_index=True
    )
    .drop_duplicates(
        subset=group_columns,
        keep="first"
    )
    .sort_values(group_columns)
    .reset_index(drop=True)
)

state_year_cases_long.head()

,year,state,ConditionName,cases,records,reporting_type
0,1995,AK,Mumps,13,46,cumulative
1,1995,AK,Pertussis,1,7,cumulative
2,1995,AL,Mumps,4,48,cumulative
3,1995,AL,Pertussis,38,48,cumulative
4,1995,AR,Measles,2,88,cumulative


### Confirm measles count for QC

In [13]:
state_year_cases_long.loc[
    (state_year_cases_long["year"] == 1998) &
    (state_year_cases_long["ConditionName"] == "Measles"),
    "cases"
].sum()

np.int64(93)

In [14]:
cases_wide_df = (
    state_year_cases_long
    .pivot(
        index=["year", "state"],
        columns="ConditionName",
        values="cases"
    )
    .reset_index()
)

cases_wide_df.columns.name = None

cases_wide_df.rename(
    columns={
        "Measles": "measles_cases",
        "Mumps": "mumps_cases",
        "Pertussis": "pertussis_cases"
    },
    inplace=True
)

cases_wide_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,12.0,7.0,305.0
4,1995,CA,108.0,206.0,463.0


# lets reformat this table

In [15]:
cases_wide_df.to_csv('../app/data/tycho_cases.csv', index=False)

In [5]:
assert False

AssertionError: 

In [ ]:
import pandas as pd

tp_1 = pd.read_table('../misc/genecounts1.txt')
tp_2 = pd.read_table('../misc/genecounts2.txt')

tps_df = pd.merge(tp_1, tp_2, on=' gene')


tps_df.rename(columns={'0_x': 'tp_1', '0_y': 'tp_2', ' gene': 'gene'}, inplace=True)
tps_df = tps_df.dropna()
tps_df

,tp_1,gene,tp_2
6,1079,dnaA,1079
7,35,dnaN,4
8,35,dnaN,0
9,0,yaaA,0
10,3983,recF,0
...,...,...,...
2217,0,yidC,0
2218,0,yidC,1
2219,0,rnpA,0
2220,0,rpmH,0


In [ ]:
tps_df['diff'] = abs(tps_df['tp_1'] - tps_df['tp_2'])
tps_df

,tp_1,gene,tp_2,diff
10,3983,recF,0,3983
6,1079,dnaA,1079,0
1911,880,ackA,588,292
234,524,cysS,134,390
1603,372,ftsA,488,116
...,...,...,...,...
1106,0,flgB,0,0
1105,0,flgK,0,0
1104,0,cheR,0,0
1103,0,fliY,0,0


In [ ]:
tps_df = tps_df[(tps_df['tp_1'] > 0) & (tps_df['tp_2'] > 0)]

In [ ]:
tps_df = tps_df.sort_values(by='diff', ascending=False)
tps_df.head(10)

,tp_1,gene,tp_2,diff
12,52,gyrA,7832,7780
234,524,cysS,134,390
1911,880,ackA,588,292
940,30,prpE,249,219
11,215,gyrB,22,193
1831,80,folC,262,182
229,201,disA,51,150
1261,23,spoVS,153,130
228,166,radA,42,124
753,125,cadA,3,122


In [ ]:
import plotly.express as px

plot_df = tps_df.head(10).melt(
    id_vars="gene",
    value_vars=["tp_1", "tp_2"],
    var_name="time_point",
    value_name="counts"
)

fig = px.bar(
    plot_df,
    x="gene",
    y="counts",
    color="time_point",
    barmode="group",
    text="counts",
    title="Top 10 Genes by Count",
    labels={
        "gene": "Gene",
        "counts": "Count",
        "time_point": "Time Point",
    },
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Gene",
    yaxis_title="Count",
    legend_title="Time Point",
)

fig.show()